# Sleep Learning Engine - Drive-backed low-RAM cloud render

This notebook is the **personal / recurring-render** variant of
the public Colab notebook at `docs/cloud/low_ram_render.ipynb`.
Use it when you have a stable ambient library and a recurring
render workflow:

- The 97 normalised ambient mp3 files live in Drive and are
  mounted on every Colab session (cell 2). You upload them
  **once** and forget about them.
- The script and the background image or video change with
  every project, so you upload them fresh at the start of
  each render (cell 3).

**Recommended Drive layout** (create it once, reuse forever):

```
My Drive/
└── sleep-learning-engine/
    └── ambient/      <- 97 normalised mp3 tracks (persistent)
```

The script and background image are uploaded per-session and
are NOT stored in Drive (they change with every project).

**Runtime setup (one-time, every fresh Colab session):**
1. Runtime -> Change runtime type -> T4 GPU -> RAM amplia = On -> Save
2. Reconnect when prompted
3. Click Runtime -> Run all (Ctrl+F9)

**Cost:** free. Colab free sessions cap at 12 hours. A 6-minute
video finishes in well under 10 minutes end-to-end.

In [ ]:
# 1. Install sleep_learning_engine from the public repo. Tarball URL
# (not a git clone) so pip does not need git credentials.

# Purge pip's wheel cache first. Colab re-uses cached wheels across
# sessions, and a stale cache can serve the OLD package even after
# we push fixes to main. This is the fix for the 'I re-ran the
# notebook but the video did not change' report - the user was
# stuck on a cached wheel that did not include the legacy-toml
# fallback fix.
!pip cache purge
# --force-reinstall is belt-and-suspenders: even if cache purge
# missed something, this re-installs the wheel in place.
!pip install -q --force-reinstall https://github.com/fernandojjq/sleep-learning-engine/archive/refs/heads/main.tar.gz

# Print the installed version so the user can see whether the
# cache purge + force-reinstall actually picked up the new wheel.
# If this still shows the old version, the install silently
# skipped (network error, repo 404, etc.) and the rest of the
# notebook will use the previous code.
!pip show sleep_learning_engine | grep -E "^(Version|Location):"

# Verify the runtime actually has a GPU. The previous version only
# checked the ffmpeg encoder list, which lies: h264_nvenc can be in
# the list even when no GPU is bound to the container. The real test
# is nvidia-smi (raises if no GPU is present).
import subprocess
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"], capture_output=True, text=True, timeout=15
)
if smi.returncode == 0 and smi.stdout.strip():
    print("GPU detected:")
    print(smi.stdout)
    print("Note: NVENC does NOT use VRAM. 0.0/15.0 GB USED is normal -",
          "NVENC runs on dedicated T4 silicon, not on the GPU cores.",
          "If VRAM shows >0 GB, the encode is going through the GPU",
          "and is therefore hardware-accelerated.")
else:
    print("=" * 60)
    print("NO GPU DETECTED in this Colab runtime.")
    print("Encode will fall back to libx264 (CPU), 5-10x slower.")
    print("")
    print("Fix: Runtime -> Change runtime type -> T4 GPU -> Save.")
    print("Reconnect when prompted. Re-run this cell.")
    print("=" * 60)

nvenc = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True
).stdout
if "h264_nvenc" in nvenc:
    print("NVENC encoder: AVAILABLE")
else:
    print("NVENC encoder: NOT in ffmpeg build (libx264 fallback)")

# Replace Colab's system ffmpeg 4.4.2 with a modern static build.
# The system ffmpeg is from Ubuntu 22.04 (released 2021) and has
# known issues with the audio filter graph the studio uses
# (sidechaincompress + alimiter + concat demuxer) - the filter
# graph parser bails out with 'Stream specifier matches no
# streams' on that version. The static build is NVENC-capable,
# has every audio filter we need, and is what the rest of the
# notebook will find on PATH after this cell runs. ~80 MB
# download, ~30 s on a typical Colab connection.
import subprocess
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout
if "ffmpeg version 4." in ffmpeg_version:
    print("Upgrading ffmpeg (Colab system is 4.4.2, too old for the studio filter graph)...")
    ffmpeg_install_cmd = (
        "set -e; "
        "curl -sL https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz "
        "| tar xJ; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe; "
        "chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe"
    )
    subprocess.check_call(['bash', '-c', ffmpeg_install_cmd])
    print("ffmpeg upgraded.")
else:
    print("ffmpeg is already modern enough.")
print(subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.split('\n')[0])

# === CONFIG: edit this dict and re-run to change the render. ===
# The cell writes the values to <WORK>/.sleeplens.toml and
# exports SLEEP_LEARNING_ENGINE_HOME so the CLI finds it.
# Default values match the project's baseline (Brian voice,
# -5% rate, ambient at 0.34 = ~89% louder than the package
# Full list of Edge TTS voice ids: https://learn.microsoft.com/
CONFIG = {
    "tts_voice": "en-US-BrianNeural",  # Edge TTS voice id
    "tts_rate": "-5%",                   # Edge TTS rate (e.g. -10% slower)
    "tts_pitch": "-2Hz",                 # Edge TTS pitch (e.g. -4Hz lower)
    "ambient_volume": 0.34,             # 0.0 (silent) to 1.0 (max)
    "ambient_duck_db": 12.0,            # dB cut on the bed while voice speaks
    "pause_between_paragraphs": 0.0,     # 0 = none; 1.8 was the old buggy default
    "language": "en",                    # script + voice language
    "output_preset": "sleep_720p",       # sleep_720p or sleep_1080p or audio_only
    "hardware_accel": "auto",           # auto | nvenc | qsv | amf | libx264
    "render_threads": 1,                # 0 = auto, 1 = safest on shared Colab
}

import os
WORK = "/content/working"
os.makedirs(WORK, exist_ok=True)
config_path = f"{WORK}/.sleeplens.toml"
toml_lines = [
    f"# Generated by the notebook - edit the CONFIG dict above and re-run this cell.",
]
for k, v in CONFIG.items():
    if isinstance(v, str):
        toml_lines.append(f'{k} = "{v}"')
    elif isinstance(v, bool):
        toml_lines.append(f"{k} = {str(v).lower()}")
    else:
        toml_lines.append(f"{k} = {v}")
with open(config_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(toml_lines) + "\n")

print(f"Config written to {config_path}")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

# Pin the project root to the work dir so load_settings finds the toml.
os.environ["SLEEP_LEARNING_ENGINE_HOME"] = WORK

# VOICE / RATE etc. are also exposed as plain variables for the
# render cell to use in shell commands without re-reading the toml.
VOICE = CONFIG["tts_voice"]

In [ ]:
# 2. Mount Google Drive and seed the ambient library into the
# writable working directory.

# Two sources, in order:
#   (a) The 97 royalty-free tracks bundled with the package
#       (always present, copied to the work dir first).
#   (b) The user's personal library in Drive, if mounted
#       (overrides bundled tracks of the same filename, adds
#       any extras the user dropped in).
# This way the render always has music, even on a fresh Colab
# VM with no Drive setup. The user can extend the library by
# uploading to Drive, without having to ship their own copy
# of the bundled tracks.
from google.colab import drive
import os, shutil, glob
from importlib.resources import files as _pkg_files

# Path conventions (edit if you put things elsewhere):
DRIVE_ROOT = "/content/drive/MyDrive/sleep-learning-engine"
AMBIENT_SRC = f"{DRIVE_ROOT}/ambient"
WORK = "/content/working"
ASSETS = f"{WORK}/assets"
AMBIENT_DST = f"{ASSETS}/ambient"

# Mount. force_remount=False avoids the OAuth popup when the
# session was started with Drive already attached. The first
# time in a session this WILL pop up a Google login.
drive.mount('/content/drive', force_remount=False)

for d in (ASSETS, AMBIENT_DST, f"{WORK}/output"):
    os.makedirs(d, exist_ok=True)

# (a) Copy the bundled ambient from the installed package.
bundled_copied = 0
try:
    bundled = _pkg_files("sleep_learning_engine").joinpath("assets", "ambient")
    if bundled.is_dir():
        for entry in bundled.iterdir():
            if entry.name.startswith("."):
                continue
            dst = os.path.join(AMBIENT_DST, entry.name)
            if not os.path.exists(dst):
                with entry.open("rb") as src_fh, open(dst, "wb") as dst_fh:
                    shutil.copyfileobj(src_fh, dst_fh)
                bundled_copied += 1
except Exception as e:
    print(f"(Bundled ambient copy skipped: {e})")

# (b) Overlay the user's Drive library on top of the bundled
#     one. Same filename -> Drive wins; new files -> added.
drive_copied = 0
drive_source_present = os.path.exists(AMBIENT_SRC)
if drive_source_present:
    for ext in ("mp3", "ogg", "wav", "flac", "m4a", "aac"):
        for src in glob.glob(f"{AMBIENT_SRC}/*.{ext}"):
            dst = os.path.join(AMBIENT_DST, os.path.basename(src))
            shutil.copy2(src, dst)  # overwrite or add
            drive_copied += 1
else:
    print(f"INFO: {AMBIENT_SRC} not found. Using bundled ambient only.")
    print("  To extend the library, create that folder in Drive and re-run this cell.")

total = len(glob.glob(f"{AMBIENT_DST}/*"))
print(f"Ambient library: {total} tracks total")
print(f"  Bundled (from package): {bundled_copied} new this session")
print(f"  Drive (your overrides): {drive_copied} files")
print(f"  Resolved path: {AMBIENT_DST}")

# Sanity check: the bundled ambient MUST exist in the installed
# wheel. If bundled_copied is 0 and the package has no ambient,
# the wheel is stale (pip cache served an old build) or the
# package_data in pyproject.toml is missing the glob.
try:
    from importlib.resources import files as _dbg_files
    _dbg = _dbg_files("sleep_learning_engine").joinpath("assets", "ambient")
    _dbg_n = len([e for e in _dbg.iterdir() if e.name.endswith(".mp3")]) if _dbg.is_dir() else 0
    print(f"  Package ambient check: {_dbg_n} mp3s at {_dbg}")
    if _dbg_n == 0:
        print("  WARNING: package wheel does NOT contain ambient tracks.")
        print("  The render will be silent. Force-reinstall in cell 1 did not pick up the new wheel.")
except Exception as e:
    print(f"  (Package ambient check failed: {e})")

In [ ]:
# 3. Upload the script and the background image (or short video) for
# THIS render. The script and background change with every project,
# so the only thing that is persistent across renders is the ambient
# library (handled in cell 2 via Drive).
from google.colab import files
import os, glob, shutil

print("Upload the script text file (.txt) and the background image or video for THIS render...")
uploaded = files.upload()
for name, data in uploaded.items():
    target = f"{ASSETS}/{name}"
    with open(target, "wb") as fh:
        fh.write(data)
print(f"Uploaded: {sorted(uploaded.keys())}")

# Resolve the paths. CRITICAL: pick the file the user JUST uploaded
# in this cell run, not 'the first .txt alphabetically' in assets/.
# Cell 6 cleans assets/ between renders, but if the user re-runs
# cell 3 without cleaning first, or if they upload a file whose
# name sorts AFTER a previous one, the alphabetical fallback
# would silently pick the wrong file. The user reported 3 consecutive
# renders producing the same video because the old 'Most developers
# use large language.txt' sorted before 'nuevo.txt' (M < n) and
# cell 3 picked the old one every time. Picking from the upload
# dict directly fixes this.
SCRIPT = ""
for name in sorted(uploaded.keys()):
    if name.lower().endswith(".txt"):
        SCRIPT = f"{ASSETS}/{name}"
        break
if not SCRIPT:
    for pat in ("*.txt",):
        matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
        if matches:
            SCRIPT = matches[0]
            break

BG_IMAGE = ""
BG_VIDEO = ""
for name in sorted(uploaded.keys()):
    if name.lower().endswith((".mp4", ".mov", ".webm")) and not BG_VIDEO:
        BG_VIDEO = f"{ASSETS}/{name}"
    elif name.lower().endswith((".jpg", ".jpeg", ".png")) and not BG_IMAGE:
        BG_IMAGE = f"{ASSETS}/{name}"
if not BG_VIDEO:
    for pat in ("*.mp4", "*.mov", "*.webm"):
        matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
        if matches:
            BG_VIDEO = matches[0]
            break
if not BG_IMAGE:
    for pat in ("*.jpg", "*.jpeg", "*.png"):
        matches = sorted(glob.glob(f"{ASSETS}/{pat}"))
        if matches:
            BG_IMAGE = matches[0]
            break

print(f"Script:      {SCRIPT or '<missing>'}")
print(f"Background:  {BG_IMAGE or BG_VIDEO or '<missing>'}")
print(f"  is video:  {bool(BG_VIDEO)}")

if not SCRIPT:
    raise SystemExit("Need a .txt script. Upload one in this cell.")
if not (BG_IMAGE or BG_VIDEO):
    raise SystemExit("Need a background image or video. Upload one in this cell.")

# Optional: pull the script/image from Drive instead. Uncomment to use.
# Useful when you have a recurring topic (same script, new ambient).
# SCRIPT = f"{DRIVE_ROOT}/scripts/prueba.txt"
# BG_IMAGE = f"{DRIVE_ROOT}/images/logo_sleeping_dev.jpeg"

In [ ]:
# 4. Render. The ffmpeg encode uses real NVENC on the T4. A 6-minute
# 1080p video finishes in 1-2 minutes of GPU time.
import os, subprocess, sys

# On Colab, ffmpeg spawned as a subprocess from Python does NOT
# inherit the LD_LIBRARY_PATH that points at the NVIDIA driver
# libraries. Without them, the NVENC encoder fails silently or
# falls back to a software emulation that is 5-10x slower. Patch
# the env before the subprocess.run call.
NVIDIA_LIB_PATHS = [
    "/usr/lib64-nvidia",          # Colab default
    "/usr/lib/x86_64-linux-gnu",  # Debian / Ubuntu
    "/usr/local/cuda/lib64",      # Manual CUDA install
]
nvidia_path = next((p for p in NVIDIA_LIB_PATHS if os.path.exists(p)), None)
if nvidia_path:
    print(f"NVIDIA libs: patching LD_LIBRARY_PATH to include {nvidia_path}")
    os.environ["LD_LIBRARY_PATH"] = nvidia_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

OUT_STEM = "sleep_learning_engine-drive"
cmd = [
    "python", "-m", "sleep_learning_engine", "render",
    "--script", SCRIPT,
    "--output-stem", OUT_STEM,
]
if os.path.exists(BG_IMAGE):
    cmd += ["--background-image", BG_IMAGE]

env = {
    **os.environ,
    "TMP": WORK, "TEMP": WORK,
    # Point the pipeline at /content/working so the resolved
    # ambient_dir (which is where cell 2 copied the 97 mp3s)
    # and output_dir (which is where this notebook expects the
    # MP4 to land) are both correct. Without this env var the
    # auto-detect walks up from the pip-installed package and
    # lands in /usr/local/lib/python3.12/dist-packages/.../assets/ambient
    # (empty) and the same for output (not writable by the user).
    "SLEEP_LEARNING_ENGINE_HOME": WORK,
    # Lets the user swap voice by editing the single VOICE line
    # at the top of cell 1, no config file required.
    "SLEEP_LEARNING_ENGINE_TTS_VOICE": VOICE,
}
if nvidia_path:
    env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

# Defensive: re-write the toml from the CONFIG dict right before
# the render runs. Idempotent - protects against 'I edited CONFIG
# in cell 1 but the toml never refreshed' and 'I only re-ran this
# cell and CONFIG was stale'.
# If cell 1 was skipped entirely, CONFIG is undefined and we fail
# with a clear error pointing back at cell 1 instead of silently
# using package defaults (which is what produced 'the video did
# not change' on the last debug round).
if "CONFIG" not in globals():
    raise SystemExit(
        "CONFIG dict is not defined. Re-run cell 1 to write the "
        "toml and export SLEEP_LEARNING_ENGINE_HOME, then re-run "
        "this cell."
    )
toml_lines = [
    "# Re-emitted by cell 4 right before the render runs.",
]
for k, v in CONFIG.items():
    if isinstance(v, str):
        toml_lines.append(f'{k} = "{v}"')
    elif isinstance(v, bool):
        toml_lines.append(f"{k} = {str(v).lower()}")
    else:
        toml_lines.append(f"{k} = {v}")
config_path = f"{WORK}/.sleeplens.toml"
with open(config_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(toml_lines) + "\n")

# Cat the toml so the user can see exactly what the render uses.
print(f"Project root : {WORK}")
print(f"Config toml  : {config_path}")
print("Active settings that the CLI will read:")
with open(config_path, "r", encoding="utf-8") as fh:
    print(fh.read(), end="")
print(f"Voice        : {VOICE}")
print(f'Hardware     : {"NVENC" if nvidia_path else "libx264"}')
print("Running      :", " ".join(cmd))
print("-" * 60)
print("STREAMING OUTPUT (will return when the render finishes):")
print("-" * 60)

# subprocess.run with capture_output=True. ffmpeg's stdout and
# stderr are buffered until ffmpeg exits, then we print them.
# No Popen, no for-line loop, no threads, no deadlock possible.
# A hard timeout (4 hours) protects against an invisible hang;
# if the timeout fires, the cell surfaces TimeoutExpired instead
# of sitting at 'Encoding: 100%' forever.
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=env,
    cwd=WORK,
    timeout=4 * 60 * 60,
)
# Print the captured output in one go. The user does not need
# a live percentage; the file appearing in cell 5 is the signal.
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr[-4000:])
print("-" * 60)
if result.returncode != 0:
    raise SystemExit(f"Render failed with exit code {result.returncode}. See stderr above.")
print("Render finished.")

In [ ]:
# 5. Locate the final MP4 and download it to your machine.
import glob, os
from google.colab import files

# Cell 4 set SLEEP_LEARNING_ENGINE_HOME=WORK so the pipeline's
# resolved output_dir is {WORK}/output, and the MP4 lands there
# directly. (Earlier iterations of this notebook had to glob
# both {WORK}/output/ and the package's /usr/local/lib/.../output
# because the project root was not set and the auto-detect
# pointed at the pip install path.)

# IMPORTANT: the CLI calls paths.unique_output() which NEVER
# overwrites - the second render goes to OUT_STEM-1.mp4, the
# third to OUT_STEM-2.mp4, etc. The previous version of this
# cell used glob('OUT_STEM.mp4') which only matched the FIRST
# render's file. On every re-run, the user downloaded the OLD
# MP4 and assumed the new render was identical to the old one
# ('el video no cambio, mismo peso').
# Fix: match OUT_STEM*.mp4 and pick the newest by mtime.
import time
candidates = sorted(glob.glob(f"{WORK}/output/{OUT_STEM}*.mp4"), key=os.path.getmtime, reverse=True)
if not candidates:
    raise SystemExit("No MP4 was produced. Check the render cell for errors.")
output = candidates[0]
# Show all candidates so the user can see the previous renders
# are still there (not overwritten) and which one we picked.
print("MP4s in output dir (newest first):")
for c in candidates:
    mtime = time.strftime("%H:%M:%S", time.localtime(os.path.getmtime(c)))
    size_mb = os.path.getsize(c) / 1e6
    marker = " <- downloading this one" if c == output else ""
    print(f"  {mtime}  {size_mb:6.1f} MB  {os.path.basename(c)}{marker}")
print(f"Final video: {output}")
print(f"Size: {os.path.getsize(output)/1e6:.1f} MB")
files.download(output)

In [ ]:
# 6. Clean up the per-render uploads so the next video starts fresh.
# Cell 3 picks the FIRST .txt / .jpg / .png / .mp4 in
# assets/ alphabetically. If the previous render's files are
# still there, the new run would silently pick up the old
# Safe to run: it only touches per-render inputs in
import glob, os

WORK = "/content/working"
ASSETS = f"{WORK}/assets"
patterns = (
    f"{ASSETS}/*.txt",
    f"{ASSETS}/*.jpg",
    f"{ASSETS}/*.jpeg",
    f"{ASSETS}/*.png",
    f"{ASSETS}/*.mp4",
    f"{ASSETS}/*.mov",
    f"{ASSETS}/*.webm",
)
removed = 0
for pat in patterns:
    for p in glob.glob(pat):
        try:
            os.unlink(p)
            removed += 1
        except FileNotFoundError:
            pass
print(f"Cleaned {removed} stale input file(s) from {ASSETS}/.")
print("Next render: re-run cells 3 -> 4 -> 5 with new script and image.")

## Why this notebook and not the public one

The public `docs/cloud/low_ram_render.ipynb` asks you to re-upload
all 96 ambient mp3 files at the start of every session. If you
render more than once a week, that 5-10 min upload is friction
this notebook removes by mounting Drive and pulling the library
from there.

Both notebooks run the same `sleep_learning_engine` pipeline and
produce identical MP4s. Use whichever matches your workflow.

## When to use Kaggle instead

If you find yourself running this notebook many times per week,
and you want guaranteed T4 GPU access (Colab free is best-effort,
can be 0/15 GB during peak hours), the Kaggle notebook at
`docs/cloud/kaggle_render.ipynb` is the right answer. Kaggle
gives 2x T4 (30 GB VRAM) and 30 GPU hours/week. The dataset path
is different: you upload the mp3 files as a Kaggle dataset at
`/kaggle/input/<slug>/` instead of Google Drive.